# A micrograd neuron, via SKaiNET's NN DSL

Same tribute to Andrej Karpathy's [micrograd](https://github.com/karpathy/micrograd) as [`MicrogradNeuron.ipynb`](./MicrogradNeuron.ipynb), but built with SKaiNET's high-level neural-network DSL (`sequential<T, V> { input(n); dense(m); activation { … } }`) instead of the low-level DAG DSL.

Original snippet:

```python
from micrograd import nn
n = nn.Neuron(2)
x = [Value(1.0), Value(-2.0)]
y = n(x)
dot = draw_dot(y)
```

**How rendering works.** A `sequential { … }` builder produces a `Module<T, V>`, not a symbolic DAG — there's no graph to render until a forward pass actually executes ops. The notebook-side `Module<T, V>.asDot(input)` helper runs one forward pass under a *recording* `DefaultGraphExecutionContext` (with shape-only `VoidTensorOps`), captures every op into an `ExecutionTape`, lowers the tape to a `ComputeGraph`, and renders DOT via SKaiNET's `toGraphviz` exporter. This is the same gradient-tracer machinery that SKaiNET's autograd backprop relies on — visualizing here exercises the recording path that backward passes will reuse.

**Prerequisite:** run `./gradlew :kotlin-notebook:publishToMavenLocal` from the repo root, then restart the kernel.

In [ ]:
USE {
    repositories {
        mavenLocal()
    }
    dependencies {
        implementation("sk.ainet.app:kotlin-notebook:0.25.0")
    }
}

## Build the neuron with the NN DSL

`nn.Neuron(2)` is one fully-connected layer from 2 inputs to 1 output, followed by an activation. micrograd defaults to `tanh`; SKaiNET's NN DSL ships `relu`, `gelu`, `elu`, `leakyRelu`, `silu`, `sigmoid`, `softmax`, `logSoftmax` — `sigmoid` is the closest sigmoidal.

In [ ]:
import sk.ainet.lang.types.FP32

val ctx = DefaultNeuralNetworkExecutionContext()

// n = nn.Neuron(2)
val n = sequential<FP32, Float> {
    input(2)
    dense(1)
    activation { it.sigmoid() }       // micrograd uses tanh; sigmoid is the closest in SKaiNET's NN DSL
}

// x = [Value(1.0), Value(-2.0)]
val x = tensor<FP32, Float>(ctx, FP32::class) {
    tensor { shape(1, 2) { fromArray(floatArrayOf(1.0f, -2.0f)) } }
}

// dot = draw_dot(y)
n.asDot(x)